# 쿼리가 30분째 안 끝나는데요?

분석 쿼리가 느리고, 보고 싶은 데이터는 DW에 없고, 팀마다 구독자 수가 다르다 — 세 문제는 모두
**데이터가 놓인 '자리'의 문제**입니다. 이 노트북은 가상의 구독형 앱 데이터로 raw → staging →
dimension/fact → mart 네 층을 직접 쌓아 보면서, 같은 질문("일별 활성 구독자 수와 매출")이
raw 와 마트에서 어떻게 다르게 답해지는지 확인합니다.

- **실습 1. raw에서 바로 뽑기** — 다섯 CSV를 한 쿼리로 조인해 답을 구하고 시간과 길이를 잽니다.
- **실습 2. staging** — 소스마다 뷰 하나씩, 이름·타입·중복·시스템 이벤트만 정리합니다.
- **실습 3. dimension / fact** — 유저 일별 스냅샷 디멘션과 구독 이벤트·일별 활성 구독 팩트 두 개를 만듭니다.
- **실습 4. mart** — 파티션 하나씩 만드는 배치로 90일을 채우고 같은 질문을 세 줄로 답합니다.
- **실습 5. 파티션 재적재** — 같은 날짜를 두 번 돌려 멱등성을 확인하고 유니크 키 테스트를 겁니다.
- **실습 6. 늦게 도착한 이벤트** — 배치 시점별 숫자 차이를 보고 3일 재적재 창을 적용합니다.

## 0. 준비

아래 셀이 의존성을 설치합니다. 데이터가 없다면 먼저 터미널에서 `python generate_data.py` 를 실행해 `data/` 폴더의 CSV 다섯 개를 만들어 주세요.

In [1]:
# 의존성 설치 (이미 설치돼 있으면 빠르게 넘어갑니다)
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import datetime as dt

import duckdb
import pandas as pd

con = duckdb.connect()   # 인메모리 연결 — 노트북 전체에서 재사용

---
## 실습 1. raw에서 바로 뽑기 — 30분짜리 쿼리의 축소판

마트 없이 raw 파일만으로 "날짜별·플랫폼별·국가별 활성 구독자 수와 그날 매출(USD)"에 답해 봅니다.
활성 구독자는 구매·갱신 시점부터 기간(체험은 체험 기간)만큼 유지되고, 해지·환불이 있으면 그날로 끝납니다.
중복 제거, 통화 정리, 기간 펼치기, 스냅샷 조인, 환율 조인이 전부 한 쿼리 안에 들어갑니다.

In [3]:
raw_sql = """
WITH sub AS (                     -- 구독 이벤트: 중복 제거 + 이름·통화 정리 + SKU 기간 붙이기
  SELECT s.user_id, LOWER(s.event_type) AS event_type, s.sku_id,
         CAST(s.event_time AS TIMESTAMP) AS event_time,
         s.price_local, UPPER(s.currency) AS currency,
         CAST(CASE WHEN s.is_trial THEN k.trial_days ELSE k.duration_days END AS INTEGER) AS period_days
  FROM read_csv_auto('data/subscription_events.csv') s
  JOIN read_csv_auto('data/sku.csv') k USING (sku_id)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY s.sub_event_id ORDER BY s.received_time) = 1
),
periods AS (                      -- 구매·갱신 1건 = 구독 기간 1개
  SELECT user_id, CAST(event_time AS DATE) AS start_date,
         CAST(event_time AS DATE) + period_days - 1 AS end_date
  FROM sub WHERE event_type IN ('purchase', 'renewal')
),
ends AS (                         -- 해지·환불은 그날로 종료
  SELECT user_id, MIN(CAST(event_time AS DATE)) AS end_date
  FROM sub WHERE event_type IN ('cancel', 'refund') GROUP BY user_id
),
expanded AS (                     -- 기간을 날짜 하나하나로 펼치기
  SELECT user_id, CAST(UNNEST(generate_series(start_date, end_date, INTERVAL 1 DAY)) AS DATE) AS date
  FROM periods
),
active AS (
  SELECT DISTINCT x.user_id, x.date
  FROM expanded x LEFT JOIN ends e USING (user_id)
  WHERE (e.end_date IS NULL OR x.date < e.end_date) AND x.date <= DATE '2026-06-29'
),
snap AS (                         -- 그날 기준 유저 속성
  SELECT snapshot_date, user_id, platform, UPPER(country) AS country
  FROM read_csv_auto('data/user_snapshots.csv')
),
revenue AS (                      -- 그날 결제액을 USD 로 환산 (환불은 음수)
  SELECT CAST(s.event_time AS DATE) AS date, s.user_id,
         SUM(s.price_local * r.rate_to_usd) AS revenue_usd
  FROM sub s
  JOIN read_csv_auto('data/exchange_rates.csv') r
    ON r.date = CAST(s.event_time AS DATE) AND r.currency = s.currency
  GROUP BY 1, 2
)
SELECT a.date, sn.platform, sn.country,
       COUNT(DISTINCT a.user_id)        AS active_subscribers,
       COALESCE(SUM(rv.revenue_usd), 0) AS revenue_usd
FROM active a
JOIN snap sn ON sn.user_id = a.user_id AND sn.snapshot_date = a.date
LEFT JOIN revenue rv ON rv.user_id = a.user_id AND rv.date = a.date
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

In [4]:
t0 = time.time()
raw_result = con.execute(raw_sql).df()
print(f"쿼리 {len(raw_sql.strip().splitlines())}줄 · {time.time() - t0:.2f}초 · 결과 {len(raw_result):,}행")
raw_result.tail()

쿼리 47줄 · 0.15초 · 결과 885행


,date,platform,country,active_subscribers,revenue_usd
880,2026-06-29,ios,BR,26,7.259845
881,2026-06-29,ios,DE,11,0.000000
882,2026-06-29,ios,JP,29,4.970250
883,2026-06-29,ios,KR,72,43.043700
884,2026-06-29,ios,US,32,0.000000


노트북 규모라 몇 초 안에 끝나지만, 주목할 부분은 쿼리의 모양입니다. 이 쿼리를 질문이 올 때마다 다시 짜야 하고,
다른 분석가는 조금 다르게 짭니다(예: `revenue` 에서 환불을 빼면 매출 숫자가 달라집니다). 정의가 쿼리 안에 흩어져 있는 것이 문제입니다.

---
## 실습 2. staging — 소스마다 하나씩 정리하기

스테이징은 소스 파일 하나에 뷰 하나. 이름·타입·중복·잡음만 정리하고 **비즈니스 로직은 넣지 않습니다**.

In [5]:
con.execute("""
CREATE OR REPLACE VIEW stg_events AS
SELECT event_id, user_id, event_name, session_id,
       CAST(event_time AS TIMESTAMP)    AS event_time,
       CAST(received_time AS TIMESTAMP) AS received_time,
       CAST(event_time AS DATE)         AS event_date,
       CAST(received_time AS DATE)      AS received_date,
       properties
FROM read_csv_auto('data/events.csv')
WHERE user_id IS NOT NULL
  AND event_name NOT LIKE 'system%'                                            -- 시스템 이벤트 제외
  AND event_name <> 'heartbeat'
QUALIFY ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY received_time) = 1;  -- 재전송 중복 제거

CREATE OR REPLACE VIEW stg_subscription_events AS
SELECT sub_event_id, user_id, LOWER(event_type) AS event_type, sku_id, is_trial,
       CAST(event_time AS TIMESTAMP)    AS event_time,
       CAST(received_time AS TIMESTAMP) AS received_time,
       CAST(event_time AS DATE)         AS event_date,
       CAST(received_time AS DATE)      AS received_date,
       price_local, UPPER(currency) AS currency
FROM read_csv_auto('data/subscription_events.csv')
QUALIFY ROW_NUMBER() OVER (PARTITION BY sub_event_id ORDER BY received_time) = 1;
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM stg_subscription_events").df()

,rows,users
0,1597,1008


In [6]:
con.execute("""
CREATE OR REPLACE VIEW stg_user_snapshots AS
SELECT snapshot_date, user_id, LOWER(platform) AS platform, UPPER(country) AS country,
       app_version, install_date
FROM read_csv_auto('data/user_snapshots.csv');

CREATE OR REPLACE VIEW stg_sku AS
SELECT sku_id, duration_days, trial_days, base_price_usd FROM read_csv_auto('data/sku.csv');

CREATE OR REPLACE VIEW stg_exchange_rates AS
SELECT date, UPPER(currency) AS currency, rate_to_usd FROM read_csv_auto('data/exchange_rates.csv');
""")
con.execute("""
SELECT (SELECT COUNT(*) FROM read_csv_auto('data/events.csv')) AS raw_rows,
       (SELECT COUNT(*) FROM stg_events)                        AS stg_rows
""").df()

,raw_rows,stg_rows
0,95313,86040


raw 행 수와 스테이징 행 수의 차이가 중복과 시스템 이벤트의 양입니다. 실습 1의 쿼리는 이 정리를 매번 쿼리 안에서 하고 있었습니다.

---
## 실습 3. dimension과 fact — 설명하는 표와 측정하는 표

디멘션은 유저 일별 스냅샷과 SKU, 팩트는 두 개입니다. 구독 이벤트 한 건이 한 행인 `fact_subscription_events`(거래 팩트)와,
유저가 구독 중인 날마다 한 행인 `fact_subscriber_daily`(일별 스냅샷 팩트). 두 번째 팩트가 semi-additive 한 '활성 구독자 수'를 만듭니다.

In [7]:
con.execute("""
-- dim: 유저 × 날짜 스냅샷 — 천천히 바뀌는 속성은 그냥 매일 찍어 둔다
CREATE OR REPLACE TABLE dim_user_daily AS
SELECT snapshot_date AS date, user_id, platform, country, app_version, install_date,
       snapshot_date - install_date AS days_since_install
FROM stg_user_snapshots;

CREATE OR REPLACE VIEW dim_user_latest AS                 -- '지금 상태'가 필요할 때
SELECT * FROM dim_user_daily
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) = 1;

CREATE OR REPLACE TABLE dim_sku AS SELECT * FROM stg_sku;
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users, MAX(days_since_install) AS max_days FROM dim_user_daily").df()

,rows,users,max_days
0,136213,3000,89


In [8]:
con.execute("""
-- fact 1: 구독 이벤트 (grain: one row per 구매·갱신·해지·환불 이벤트)
CREATE OR REPLACE TABLE fact_subscription_events AS
SELECT s.sub_event_id, s.event_date, s.received_date, s.user_id, s.sku_id, s.event_type, s.is_trial,
       s.event_time, s.price_local, s.currency,
       s.price_local * r.rate_to_usd AS price_usd,                            -- 환율 조인
       CAST(CASE WHEN s.is_trial THEN k.trial_days ELSE k.duration_days END AS INTEGER) AS period_days
FROM stg_subscription_events s
LEFT JOIN stg_exchange_rates r ON r.date = s.event_date AND r.currency = s.currency
JOIN dim_sku k USING (sku_id);
""")
con.execute("""
SELECT event_type, COUNT(*) AS n, ROUND(SUM(price_usd), 2) AS usd
FROM fact_subscription_events GROUP BY 1 ORDER BY 1
""").df()

,event_type,n,usd
0,cancel,26,0.00
1,purchase,1405,6074.05
2,refund,13,-126.14
3,renewal,153,915.74


In [9]:
con.execute("""
-- fact 2: 일별 활성 구독 스냅샷 (grain: one row per 구독 중인 유저 × 날짜)
CREATE OR REPLACE TABLE fact_subscriber_daily AS
WITH periods AS (
  SELECT user_id, sku_id, is_trial, event_date AS start_date,
         event_date + period_days - 1 AS end_date
  FROM fact_subscription_events WHERE event_type IN ('purchase', 'renewal')
),
ends AS (
  SELECT user_id, MIN(event_date) AS end_date
  FROM fact_subscription_events WHERE event_type IN ('cancel', 'refund') GROUP BY user_id
),
expanded AS (
  SELECT user_id, sku_id, is_trial,
         CAST(UNNEST(generate_series(start_date, end_date, INTERVAL 1 DAY)) AS DATE) AS date
  FROM periods
)
SELECT x.date, x.user_id, x.sku_id, x.is_trial
FROM expanded x LEFT JOIN ends e USING (user_id)
WHERE (e.end_date IS NULL OR x.date < e.end_date) AND x.date <= DATE '2026-06-29'
QUALIFY ROW_NUMBER() OVER (PARTITION BY x.date, x.user_id ORDER BY x.is_trial) = 1;   -- 유니크 키 (date, user_id)
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users, MIN(date) AS d0, MAX(date) AS d1 FROM fact_subscriber_daily").df()

,rows,users,d0,d1
0,20955,1008,2026-04-01,2026-06-29


`fact_subscription_events` 의 유니크 키는 `sub_event_id`, `fact_subscriber_daily` 의 유니크 키는 `(date, user_id)` 입니다. grain 선언이 코드로 내려온 모습입니다.

한 가지 단순화: 해지·환불일을 유저당 최초 한 번(MIN)으로 잡았기 때문에 해지 후 다시 구독하는 유저는 이 팩트에서 사라집니다. 실습 데이터에는 재구독이 없어 결과가 같지만, 실무에 옮길 때는 해지를 직전 구독 기간에만 매칭하도록 보완해야 합니다.


---
## 실습 4. mart — 같은 질문을 세 줄로

두 팩트를 유저 디멘션과 같은 날짜로 조인해 `날짜 × 플랫폼 × 국가 × SKU × 체험 여부(is_trial)` 단위로 집계해 둡니다. 본문 4.2.3에서 디멘션으로 선언한 체험/정식 구분까지 그대로이고, 4.2.4의 구독 연속 횟수(streak)는 단순화를 위해 뺐습니다 — 팩트에 컬럼을 추가하며 직접 확장해 보세요.
마트는 **파티션(날짜) 하나씩** 만드는 함수로 작성합니다 — 매일 새벽 도는 배치의 축소판입니다.

In [10]:
MART_SQL = """
INSERT INTO mart_subscription_daily
WITH ev AS (                                  -- 그날의 구독 이벤트
  SELECT f.event_date AS date, d.platform, d.country, f.sku_id, f.is_trial,
         f.event_type, f.price_usd, NULL AS active_user
  FROM fact_subscription_events f
  JOIN dim_user_daily d ON d.user_id = f.user_id AND d.date = f.event_date
  WHERE f.event_date = DATE '{partition_date}' {received_filter}
),
act AS (                                      -- 그날의 활성 구독자
  SELECT a.date, d.platform, d.country, a.sku_id, a.is_trial,
         NULL AS event_type, 0 AS price_usd, a.user_id AS active_user
  FROM fact_subscriber_daily a
  JOIN dim_user_daily d ON d.user_id = a.user_id AND d.date = a.date
  WHERE a.date = DATE '{partition_date}'
)
SELECT date, platform, country, sku_id, is_trial,
       COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS new_count,
       COUNT(CASE WHEN event_type = 'renewal'  THEN 1 END) AS renewal_count,
       COUNT(CASE WHEN event_type = 'cancel'   THEN 1 END) AS cancel_count,
       COUNT(CASE WHEN event_type = 'refund'   THEN 1 END) AS refund_count,
       SUM(price_usd)                                      AS revenue_usd,
       COUNT(DISTINCT active_user)                         AS active_subscribers
FROM (SELECT * FROM ev UNION ALL SELECT * FROM act)
GROUP BY ALL
"""

In [11]:
con.execute("""
CREATE OR REPLACE TABLE mart_subscription_daily (
  date DATE, platform VARCHAR, country VARCHAR, sku_id VARCHAR, is_trial BOOLEAN,
  new_count BIGINT, renewal_count BIGINT, cancel_count BIGINT, refund_count BIGINT,
  revenue_usd DOUBLE, active_subscribers BIGINT
)""")

def run_partition(partition_date, received_filter=""):
    """한 번의 배치 = 파티션 하나. 지우고 다시 채운다 (insert-overwrite)."""
    con.execute("DELETE FROM mart_subscription_daily WHERE date = ?", [partition_date])
    con.execute(MART_SQL.format(partition_date=partition_date, received_filter=received_filter))

start, end = dt.date(2026, 4, 1), dt.date(2026, 6, 29)
for i in range((end - start).days + 1):          # 90일치 backfill = 90번 실행
    run_partition(start + dt.timedelta(days=i))
con.execute("SELECT COUNT(*) AS rows, MIN(date) AS d0, MAX(date) AS d1 FROM mart_subscription_daily").df()

,rows,d0,d1
0,5225,2026-04-01,2026-06-29


In [12]:
mart_sql = """
SELECT date, platform, country,
       SUM(active_subscribers) AS active_subscribers, SUM(revenue_usd) AS revenue_usd
FROM mart_subscription_daily GROUP BY ALL ORDER BY ALL
"""
t0 = time.time()
mart_result = con.execute(mart_sql).df()
print(f"쿼리 {len(mart_sql.strip().splitlines())}줄 · {time.time() - t0:.3f}초 · 결과 {len(mart_result):,}행")

cmp = raw_result.merge(mart_result, on=["date", "platform", "country"], suffixes=("_raw", "_mart"))
print("raw 쿼리와 마트의 활성 구독자 수 차이(합):", int((cmp.active_subscribers_raw - cmp.active_subscribers_mart).abs().sum()))
print("raw 쿼리와 마트의 매출 차이(합, USD):", round((cmp.revenue_usd_raw - cmp.revenue_usd_mart).sum(), 2))
mart_result.tail()

쿼리 3줄 · 0.001초 · 결과 885행
raw 쿼리와 마트의 활성 구독자 수 차이(합): 0
raw 쿼리와 마트의 매출 차이(합, USD): 126.14


,date,platform,country,active_subscribers,revenue_usd
880,2026-06-29,ios,BR,26.0,7.259845
881,2026-06-29,ios,DE,11.0,0.000000
882,2026-06-29,ios,JP,29.0,4.970250
883,2026-06-29,ios,KR,72.0,43.043700
884,2026-06-29,ios,US,32.0,0.000000


같은 질문이 `SELECT … FROM mart … GROUP BY` 로 끝났고, raw 쿼리와 활성 구독자 수가 정확히 같습니다. 그런데 매출(revenue_usd)은 다릅니다 — 이번 데이터에서는 raw 쪽이 126.14 USD 더 큰데요. raw 쿼리는 해지·환불 당일을 비활성 처리해 매출 조인에서 환불의 음수 금액이 함께 빠지고, 마트는 모든 구독 이벤트를 집계해 환불이 차감된 매출을 돌려주기 때문입니다. "환불을 매출에서 빼느냐"가 팩트를 만들 때 한 번 정해진 일이라, 마트를 쓰면 이 차이 자체가 사라집니다.

주의: `SUM(active_subscribers)` 는 같은 날짜 안(플랫폼·국가·SKU를 합칠 때)에서만 맞는 계산입니다. 날짜를 가로질러 더하면 semi-additive 함정에 빠집니다.

---
## 실습 5. 파티션 하나 다시 적재하기 — 멱등성과 유니크 키 테스트

6월 14일 데이터에 문제가 있었다고 가정하고 그 날짜만 다시 만듭니다. `run_partition` 은 지우고 다시 채우므로 몇 번을 실행해도 결과가 같습니다.

In [13]:
target = dt.date(2026, 6, 14)
before = con.execute("SELECT COUNT(*), ROUND(SUM(revenue_usd), 2) FROM mart_subscription_daily WHERE date = ?", [target]).fetchone()
run_partition(target); run_partition(target)          # 두 번 실행
after  = con.execute("SELECT COUNT(*), ROUND(SUM(revenue_usd), 2) FROM mart_subscription_daily WHERE date = ?", [target]).fetchone()
print("두 번 실행 전/후:", before, after)                 # 같아야 한다

# 비교: DELETE 없이 INSERT만 하면?
con.execute(MART_SQL.format(partition_date=target, received_filter=""))
dup = con.execute("""
SELECT date, platform, country, sku_id, is_trial, COUNT(*) AS n
FROM mart_subscription_daily GROUP BY ALL HAVING COUNT(*) > 1
""").df()
print(f"유니크 키 위반 {len(dup)}건")                   # 0이 아니면 테스트 실패
run_partition(target)                                 # 원상복구

두 번 실행 전/후: (70, 109.83) (70, 109.83)
유니크 키 위반 70건


---
## 실습 6. 늦게 도착한 이벤트 — 재적재 창

`received_time` 은 `event_time` 보다 최대 3일 늦습니다. 매일 '어제까지 도착한 데이터'로 '어제 파티션'만 만들면 숫자가 얼마나 빠질까요?
(활성 구독자 수는 팩트 2를 다시 만들어야 하므로 여기서는 이벤트 건수와 매출만 봅니다.)

In [14]:
def run_partition_as_of(partition_date, as_of):
    """as_of 날짜까지 도착한 이벤트만으로 partition_date 파티션을 만든다 (배치 당일 시점 재현)."""
    run_partition(partition_date, received_filter=f"AND f.received_date <= DATE '{as_of}'")

rows = []
for lag in range(0, 4):                                # 당일, +1일, +2일, +3일 시점
    run_partition_as_of(target, target + dt.timedelta(days=lag))
    n, usd = con.execute("SELECT SUM(new_count + renewal_count), ROUND(SUM(revenue_usd), 2) FROM mart_subscription_daily WHERE date = ?", [target]).fetchone()
    rows.append((f"D+{lag}", int(n), usd))
pd.DataFrame(rows, columns=["배치 시점", "구매·갱신 건수", "매출(USD)"])

,배치 시점,구매·갱신 건수,매출(USD)
0,D+0,13,82.31
1,D+1,20,96.90
2,D+2,24,104.84
3,D+3,26,109.83


D+0 시점의 숫자가 가장 작고 D+3에서야 확정됩니다. 그래서 매일 배치가 '어제 하루'가 아니라 '최근 3일'을 다시 만들도록 합니다.

In [15]:
def daily_batch(run_date, window_days=3):
    """매일 새벽 배치: 최근 window_days 파티션을 다시 만든다."""
    for lag in range(1, window_days + 1):
        run_partition(run_date - dt.timedelta(days=lag))

daily_batch(dt.date(2026, 6, 18))   # 6/15, 6/16, 6/17 파티션 재적재
run_partition(target)               # 실험했던 6/14 파티션도 원상복구

---
## 마무리

노트북에서는 뷰와 테이블을 손으로 만들었지만, 실무에서는 dbt 같은 변환 도구가 `sql/` 폴더의 파일 하나를 테이블 하나로 매핑하고,
파일 사이의 의존 관계를 읽어 실행 순서를 정하고, 유니크·not null 테스트를 YAML 한 줄로 걸어 줍니다.
이 폴더의 `sql/` 을 층별 파일로 나눠 둔 것은 그 구조를 미리 체감해 보시라는 의도입니다.